# Digital phenotyping of patients with AKI
* This notebook focusses on identifying patients with **Acute Kindney Injury (AKI).**

* How does an AKI patient look in the MIMIC-IV data? What defines them? 

* We will use the [KDIGO](https://kdigo.org/guidelines/acute-kidney-injury) criteria to define AKI. AKI is diagnosed if **any of three conditions occur**: 
    * Serum creatinine rises by ≥ 0.3 mg/dL (≥ 26.5 μmol/L) within 48 hours

    * Serum creatinine increases to ≥ 1.5 times baseline within the prior 7 days
    
    * Urine output drops below 0.5 mL/kg/h for 6 hours

**A note on data access**
* This notebook uses locally stored **MIMIC-IV Demo** data located in the `../data` folder.

* This approach is **not recommended** and is only used to enable running the notebook on Ed. 

* For your project, it is recommended to use `pandas_gbq` package to query MIMIC-IV data **directly from the cloud**. See `how-to-bigquery.ipynb` for a demo.

## Load libraries and setup environment

In [1]:
import numpy as np
import pandas as pd

# Plotting libraries
import matplotlib.pyplot as plt
plt.style.use('ggplot')
plt.rcParams['axes.facecolor'] = 'whitesmoke'
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.title_fontsize'] = 12
plt.rcParams['legend.fontsize'] = 12

# To load data from local directory of .csv files
from pathlib import Path

## Load `chartevents` table

* Patient measurements during ICU stays are recorded in `chartevents.csv.gz`.

* Each measurement is timestamped.

* Let's load the `chartevents` table and look at it.

In [2]:
# Root directory for local MIMIC-IV demo data.
data_root = Path("../data/mimic-iv-clinical-database-demo-2.2")

# Path to the icu module
icu_dir = data_root / "icu"

# Load the chartevents table
chartevents = pd.read_csv(
    icu_dir / "chartevents.csv.gz",
    parse_dates=['charttime'],
)
chartevents.head()

The table contains all possible measurements. **Which ones are for serum creatinine?**

## Criterion 1: Serum creatinine rises by ≥ 0.3 mg/dL (≥ 26.5 μmol/L) within 48 hours

### Identify relevant items

### `d_items` table

* Complex relational databases are often normalised to store data more efficiently. 

* You can see the in `chartevents` the type of measurement is recorded as `itemid`. 

* To figure out which measurements are for **serum creatinine** we need to look up the relevant IDs in the reference table.

In [3]:
# d_items table
d_items = pd.read_csv(
    icu_dir / "d_items.csv.gz",
)
d_items.head()

* You can see each `itemid` is associated with a `label`

* It also tells you which table the `itemid` caqn be found in (see `linksto`)

* Finally, `lownormalvalue` and `highnormalvalue` provide the normal reference range, where possible.

* Let's look for items with the word "creatinine" in the `label` column:

In [4]:
# Filter d_items for rows where the label contains 'creatinine' (case-insensitive)
d_items[d_items.label.str.contains('creatinine', case=False)]

* We can see that the only relevant measurement is `itemid=220615`. We can confirm it links to the `chartevents` table.

In [5]:
# From the table above, we can see that 'Creatinine (serum)' has the itemid 220615.
creatinine_itemid = 220615

### Serum creatinine measurements

In [6]:
# Filter for creatinine measurements
creatinine = chartevents[chartevents.itemid==creatinine_itemid].copy()

# Convert 'value' to numeric (float), coerce errors to NaN
creatinine.value = pd.to_numeric(creatinine.value, errors='coerce')

print(f"Found {len(creatinine)} creatinine measurements.")
creatinine.head()

### Patients with at least two measurements

* Recall that the first criterion is **Serum creatinine rises by ≥ 0.3 mg/dL (≥ 26.5 μmol/L) within 48 hours**

* This means we need at least **two measurements** for the same patient

* Let's filter our dataset to only include patients who have had two or more creatinine tests

In [7]:
# # Use groupby() and filter() to keep only patients with 2 or more measurements
# creatinine_two_or_more = creatinine.groupby('subject_id').filter(lambda x: len(x) >= 2)

# Backup code: break into separate lines for clarity
grouped = creatinine.groupby('subject_id')
creatinine_two_or_more = grouped.filter(lambda x: len(x) >= 2)
num_unique_patients = creatinine_two_or_more.subject_id.nunique()

print(f"Original number of measurements: {len(creatinine)}")
print(f"Measurements from patients with 2+ readings: {len(creatinine_two_or_more)}")
print(f"Number of unique patients with 2+ readings: {num_unique_patients}")

### Patients that satisfy criterion 1

In [8]:
def check_aki_criterion1(x):
    """
    Check if the creatinine rises by ≥ 0.3 mg/dL (≥ 26.5 μmol/L) within 48 hours:
    - Considers any pair of measurements within a 48-hour window, not just consecutive ones
    - As soon as one pair meets the criterion, return True
    """
    # Sort by charttime to ensure correct order
    x = x.sort_values(by='charttime')

    # For each measurement, look at all measurements within the last 48 hours
    for i in range(len(x)):
        # Current measurement's time and value
        current_time = x.iloc[i]['charttime']
        current_value = x.iloc[i]['value']

        prior_measurements = x[(x.charttime < current_time) & (x.charttime >= current_time - pd.Timedelta(hours=48))]

        # If any prior measurement is at least 0.3 mg/dL lower than the current measurement, criterion 1 is met
        if (current_value - prior_measurements['value'] >= 0.3).any():
            return True

    # No pair of measurements within 48 hours met the criterion
    return False

In [9]:
# Group by subject_id and apply the check_aki_criterion1 function to each patient
# Remove "include_groups=False" from the line below if using pandas >= 3.0.0
result = creatinine_two_or_more.groupby('subject_id').apply(check_aki_criterion1, include_groups=False).reset_index(name='criterion1_met')

# Identify patients meeting AKI criterion 1
aki_patients_criterion1 = result[result.criterion1_met == True].subject_id.unique().tolist()
print(f"Identified {len(aki_patients_criterion1)} patients meeting AKI criterion 1.")

## Criterion 2: Serum creatinine increases to ≥ 1.5 times baseline within the prior 7 days

* For this criterion, we again need at least **two measurements** for the same patient

* We can **re-use the `creatinine_two_or_more` dataframe** we calculated for criterion 1

* All that is left is to implement a new function to check if the criterion is met

In [10]:
def check_aki_criterion2(x):
    """
    Check if the creatinine rises to ≥ 1.5 times baseline within the prior 7 days:
    - Baseline is defined as the lowest creatinine value in the prior 7 days
    - The current measurement is at least 1.5 times the baseline value
    """
    # Sort by charttime to ensure correct order
    x = x.sort_values(by='charttime')

    # For each measurement, look at all measurements within the last 7 days
    for i in range(len(x)):
        # Current measurement's time and value
        current_time = x.iloc[i]['charttime']
        current_value = x.iloc[i]['value']

        # Get measurements in the prior 7 days
        prior_measurements = x[(x.charttime < current_time) & (x.charttime >= current_time - pd.Timedelta(days=7))]

        # If any prior measurement is at least 1.5 times lower than the current measurement, criterion 2 is met
        if (current_value >= 1.5 * prior_measurements['value']).any():
                return True

    # No pair of measurements within 7 days met the criterion
    return False

In [11]:
# Group by subject_id and apply the check_aki_criterion2 function to each patient
# Remove "include_groups=False" from the line below if using pandas >= 3.0.0
result = creatinine_two_or_more.groupby('subject_id').apply(check_aki_criterion2, include_groups=False).reset_index(name='criterion2_met')

# Identify patients meeting AKI criterion 2
aki_patients_criterion2 = result[result.criterion2_met == True].subject_id.unique().tolist()
print(f"Identified {len(aki_patients_criterion2)} patients meeting AKI criterion 2.")

## Criterion 3: Urine output drops below 0.5 mL/kg/h for 6 hours

* Finally, let's find patients that satisfy criterion 3

* Note that here we need to calculate the amount in mL **per kilogram per hour**

* This means we will need to look at the urine output measurements **AND** patient weight

* **What will be your steps?**
    * You should start with figuring out how patient's weight and urine outut are stored

### Weight

* **Which items correspond to patient's weight?**
    * It's ok to use admission weight: `itemid=226531` and `itemid=226512`

* **Which units should we use? kg or lbs?**
    * Could be a good idea to convert pounds to kilograms

* **Which table should you query to extract admission weight?**
    * Both measurements are stored in `chartevents`

In [12]:
# Filter d_items for rows where the label contains 'weight' (case-insensitive)
d_items[d_items.label.str.contains('weight', case=False)]

In [13]:
# From the table above, we can see that 'Weight (lbs)' has the itemid 226531 and 'Weight (kg)' has the itemid 226512.
weight_lbs_item_id = 
weight_kg_itemid = 

In [14]:
# Filter for weight measurements in kilograms
weight_kg = chartevents[chartevents.itemid==weight_kg_itemid].copy()

# Convert 'value' to numeric (float), coerce errors to NaN
weight_kg.value = pd.to_numeric(weight_kg.value, errors='coerce')

# Filter for weight measurements in pounds
weight_lbs = chartevents[chartevents.itemid==weight_lbs_item_id].copy()

# Convert 'value' to numeric (float), coerce errors to NaN
weight_lbs.value = pd.to_numeric(weight_lbs.value, errors='coerce')

# Convert weight in pounds to kilograms (1 lb = 0.453592 kg)
weight_lbs.value = 

# Combine weight measurements in kilograms and converted pounds
weight = pd.concat([weight_kg, weight_lbs], ignore_index=True)

print(f"Found {len(weight)} weight measurements.")
weight.head()

### Urine output
* **Which items correspond to urine output?**

    * This is where your mates with clinical background are invaluable! Turns out **just checking for the word "urine" is not enough**.

In [15]:
# Look up items that contain the word "urine"
d_items[d_items.label.str.contains('urine', case=False)]

* Actually none of these are relevant. This is why it is important to **work closely with domain experts** -- they will point you in the right direction! If you have someone with medical background in your team **make sure to leverage their knowledge**.

* Examine the items below and note the **table** in which the measurements are stored and the **units**.

In [16]:
# Here are the items you should consider looking up in the d_items table:
urine_output_itemids = [226559, 226560, 226561, 226563, 226564, 226565, 226567, 226584, 227510]
d_items[d_items.itemid.isin(urine_output_itemids)]

* A clinician can **quickly correlate many different factors** in their head and conclude whether a patient has AKI or not.

* But when working with data, we need to **explicitly break down this reasoning** into concrete steps.

* Figuring out **which measurements are relevant** to determine the urine outout is a great example of how **a clinical concept can be stored in many different source fields**.

### `outputevents` table

In [17]:
# Load the outputevents table
outputevents = pd.read_csv(
    icu_dir / "outputevents.csv.gz",
    parse_dates=['charttime'],
)
outputevents.head()

In [18]:
# Filter for relevant measurements
urine_output = 

# Convert 'value' to numeric (float), coerce errors to NaN
urine_output.value = 

print(f"Found {len(urine_output)} measurements of urine output from {urine_output.subject_id.nunique()} patients.")
urine_output.head()

* **Every one of our 100 patients** has had their urine output recorded at least once.

* Recall the criterion: **Urine output drops below 0.5 mL/kg/h for 6 hours**

* **How do we go about calculating this?**

    * From the `d_items` table we know that all urine output is **recorded in mL** so no need to convert the units.

    * One way to check if the criterion is met is to create a **sliding window of 6 hours** adding up all urine output within the window. 
    
    * The total is then divided by the patient's weight and the total number of hours (6) **to get the result in the correct units - mL/kg/h**

    

In [19]:
def check_aki_criterion3(x):
    """
    Check if urine output is < 0.5 mL/kg/h for 6 hours:
    - Requires weight information to calculate urine output per kg
    - For each measurement, look at the next 6 hours of measurements
    - If the average urine output over that period is < 0.5 mL/kg/h, criterion 3 is met
    """
    # Sort by charttime to ensure correct order
    x = x.sort_values(by='charttime')

    # Get the patient's weight (assuming the most recent weight measurement is the best estimate)
    patient_id = x.subject_id.iloc[0]
    patient_weight = 

    # For each measurement, look at the previous 6 hours of measurements
    for i in range(len(x)):
        # Current measurement's time and value
        current_time = 
        current_value = 

        # Get measurements in the previous 6 hours
        previous_measurements = 

        # Calculate total urine output over the previous 6 hours
        total_urine_output = 

        # Calculate average urine output per hour
        avg_urine_output_per_hour = 

        # Calculate urine output per kg per hour
        urine_output_per_kg_per_hour = 

        # If urine output is < 0.5 mL/kg/h, criterion 3 is met
        if urine_output_per_kg_per_hour < 0.5:
            return True

    # No period of 6 hours met the criterion
    return False

In [24]:
# Identify patients meeting AKI criterion 3

print(f"Identified {len(aki_patients_criterion3)} patients meeting AKI criterion 3.")

## Final cohort of AKI patients

* Recall that AKI is diagnosed if **any of three conditions occur**:

    * Serum creatinine rises by ≥ 0.3 mg/dL (≥ 26.5 μmol/L) within 48 hours

    * Serum creatinine increases to ≥ 1.5 times baseline within the prior 7 days
    
    * Urine output drops below 0.5 mL/kg/h for 6 hours

* **Some patients might meet more than one criterion.** Let's combine the lists of patients to derive the final cohort. 

In [25]:
print(f"We found {len(aki_patients_criterion1)} patients that meet criterion 1, {len(aki_patients_criterion2)} patients that meet criterion 2, and {len(aki_patients_criterion3)} patients that meet criterion 3.")

print(f"For example, {len([subject_id for subject_id in aki_patients_criterion3 if subject_id in aki_patients_criterion1])} patients that meet criterion 3 also meet criterion 1.")

# Combine all patients meeting any of the three criteria into a final cohort
final_cohort = 
print(f"Total number of unique patients meeting any AKI criterion: {len(final_cohort)}")